# gf_mul CNN prediction matrix generation

`gf_mul_cnn.ipynb`에서 학습한 CNN weight를 불러와 prediction matrix를 만든다.

저장 구조:

```text
predict_matrix_cnn/
  pred_cnn_{MODEL_INDEX}.h5
```

HDF5 내부 구조는 기존 LDA prediction matrix와 맞췄다.

```text
op1_hw/pred_matrix
op1_hw/true_labels
out_hw/pred_matrix
out_hw/true_labels
```

SASCA practical notebook에서는 `PRED_PATH`만 이 파일로 바꾸면 된다.

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import h5py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

In [ ]:
# =====================
# User configuration
# =====================

TRACE_H5_PATH = './gf_mul_traces.h5'
MODEL_INDEX = 0

WEIGHT_DIR = Path('weight_cnn')
PRED_DIR = Path('predict_matrix_cnn')
PRED_DIR.mkdir(exist_ok=True)

TARGETS_TO_PREDICT = ['op1_hw', 'out_hw']

# 'test': 학습에 쓰지 않은 test split만 저장
# 'all' : 전체 trace 저장
PREDICT_SPLIT = 'test'

BATCH_SIZE = 512
OUT_H5_PATH = PRED_DIR / f'pred_cnn_{MODEL_INDEX}.h5'

In [ ]:
def gf_mul_ref_python(a, b):
    a = int(a) & 0xff
    b = int(b) & 0xff
    res = 0
    for _ in range(8):
        if b & 1:
            res ^= a
        carry = a & 0x80
        a = (a << 1) & 0xff
        if carry:
            a ^= 0x1d
        b >>= 1
    return res & 0xff

HW = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)

def _first_existing_dataset(h5, candidates):
    for name in candidates:
        if name in h5 and isinstance(h5[name], h5py.Dataset):
            return h5[name][:], name
    found = []
    def visit(name, obj):
        if isinstance(obj, h5py.Dataset):
            found.append(name)
    h5.visititems(visit)
    raise KeyError(f'Could not find any of {candidates}. Existing datasets: {found[:30]}')

def load_trace_h5(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'TRACE_H5_PATH not found: {path.resolve()}')
    with h5py.File(path, 'r') as f:
        traces, trace_key = _first_existing_dataset(f, ['traces', 'trace', 'X', 'samples', 'traces_p'])
        a, a_key = _first_existing_dataset(f, ['a', 'op0', 'operand0', 'input_a', 'labels/a'])
        b, b_key = _first_existing_dataset(f, ['b', 'op1', 'operand1', 'input_b', 'labels/b'])
        try:
            v, v_key = _first_existing_dataset(f, ['v', 'out', 'output', 'result', 'labels/v', 'labels/out'])
        except KeyError:
            v = np.array([gf_mul_ref_python(x, y) for x, y in zip(a, b)], dtype=np.uint8)
            v_key = 'computed gf_mul_ref_python(a,b)'
    traces = np.asarray(traces)
    a = np.asarray(a, dtype=np.uint8).reshape(-1)
    b = np.asarray(b, dtype=np.uint8).reshape(-1)
    v = np.asarray(v, dtype=np.uint8).reshape(-1)
    n = min(len(traces), len(a), len(b), len(v))
    traces, a, b, v = traces[:n], a[:n], b[:n], v[:n]
    print('loaded:', path)
    print('trace key:', trace_key, traces.shape)
    print('a key:', a_key, 'b key:', b_key, 'v key:', v_key)
    return traces.astype(np.float32), a, b, v

def make_labels(a, b, v):
    return {
        'op0_val': a.astype(np.int64),
        'op1_val': b.astype(np.int64),
        'out_val': v.astype(np.int64),
        'op0_hw': HW[a].astype(np.int64),
        'op1_hw': HW[b].astype(np.int64),
        'out_hw': HW[v].astype(np.int64),
    }

In [ ]:
class TraceDataset(Dataset):
    def __init__(self, traces, y, indices, mean, std):
        self.traces = traces
        self.y = y.astype(np.int64)
        self.indices = np.asarray(indices)
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        x = (self.traces[idx:idx+1] - self.mean) / self.std
        return torch.from_numpy(x.astype(np.float32)), torch.tensor(self.y[idx], dtype=torch.long), idx

class SimpleGFTraceCNN(nn.Module):
    def __init__(self, trace_len, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=11, padding=5),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=9, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(16),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(64 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def predict_proba_for_target(target_name, traces, labels, indices):
    ckpt_path = WEIGHT_DIR / f'cnn_{target_name}_{MODEL_INDEX}.pt'
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Missing weight: {ckpt_path}')
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = SimpleGFTraceCNN(trace_len=ckpt['trace_len'], num_classes=ckpt['num_classes']).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    mean = ckpt['mean']
    std = ckpt['std']
    y = labels[target_name]
    ds = TraceDataset(traces, y, indices, mean, std)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    probs, ys, idxs = [], [], []
    with torch.no_grad():
        for xb, yb, ib in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            p = torch.softmax(logits, dim=1).cpu().numpy()
            probs.append(p)
            ys.append(yb.numpy())
            idxs.append(ib.numpy())
    P = np.concatenate(probs, axis=0).astype(np.float32)
    Y = np.concatenate(ys, axis=0).astype(np.int64)
    I = np.concatenate(idxs, axis=0).astype(np.int64)
    acc = float(np.mean(P.argmax(axis=1) == Y))
    print(f'[{target_name}] pred_matrix shape={P.shape}, acc={acc:.4f}')
    print(confusion_matrix(Y, P.argmax(axis=1), labels=np.arange(ckpt['num_classes'])))
    return P, Y, I, ckpt, acc

In [ ]:
traces, a, b, v = load_trace_h5(TRACE_H5_PATH)
labels = make_labels(a, b, v)

split_path = WEIGHT_DIR / f'split_indices_{MODEL_INDEX}.npz'
if split_path.exists():
    split = np.load(split_path)
    if PREDICT_SPLIT == 'test':
        pred_indices = split['test_idx']
    elif PREDICT_SPLIT == 'train':
        pred_indices = split['train_idx']
    elif PREDICT_SPLIT == 'val':
        pred_indices = split['val_idx']
    elif PREDICT_SPLIT == 'all':
        pred_indices = np.arange(len(traces))
    else:
        raise ValueError('PREDICT_SPLIT must be test/train/val/all')
else:
    print('split_indices not found; using all traces')
    pred_indices = np.arange(len(traces))

print('PREDICT_SPLIT =', PREDICT_SPLIT)
print('num pred samples =', len(pred_indices))

In [ ]:
with h5py.File(OUT_H5_PATH, 'w') as f:
    f.attrs['model_index'] = MODEL_INDEX
    f.attrs['predict_split'] = PREDICT_SPLIT
    f.attrs['trace_h5_path'] = str(TRACE_H5_PATH)
    f.attrs['format'] = 'cnn_prediction_matrix_v1'

    for target in TARGETS_TO_PREDICT:
        P, Y, I, ckpt, acc = predict_proba_for_target(target, traces, labels, pred_indices)
        g = f.create_group(target)
        g.create_dataset('pred_matrix', data=P, compression='gzip', compression_opts=4)
        g.create_dataset('true_labels', data=Y, compression='gzip', compression_opts=4)
        g.create_dataset('sample_indices', data=I, compression='gzip', compression_opts=4)
        g.attrs['acc_top1'] = acc
        g.attrs['weight_file'] = str(WEIGHT_DIR / f'cnn_{target}_{MODEL_INDEX}.pt')
        g.attrs['num_classes'] = int(P.shape[1])

print('saved prediction matrix:', OUT_H5_PATH)

In [ ]:
# Quick reload check
with h5py.File(OUT_H5_PATH, 'r') as f:
    print('file:', OUT_H5_PATH)
    print('attrs:', dict(f.attrs))
    for target in f.keys():
        P = f[target]['pred_matrix'][:]
        y = f[target]['true_labels'][:]
        print(target, P.shape, 'acc=', f[target].attrs['acc_top1'], 'recalc=', np.mean(P.argmax(axis=1)==y))